In [ ]:
# importing libraries
import pandas as pd
import numpy as np


In [ ]:
# reading all datasets
tmdb_df = pd.read_csv("../../data/processed/tmdb_clean.csv")
numbers_df = pd.read_csv("../../data/processed/the_numbers_worldwide_2026_cleaned.csv")
rotten_df= pd.read_csv("../../data/processed/rotten_tomatoes_cleaned.csv")

In [ ]:
# testing merging tmdb dataset with numbers using the title column
tmdb_numbers_df = pd.merge(
    tmdb_df,
    numbers_df,
    on="title",
    how="inner"
)

print("TMDB:", len(tmdb_df))
print("The Numbers:", len(numbers_df))
print("Merged:", len(tmdb_numbers_df))

display(tmdb_numbers_df.head())


In [ ]:
# testing merging tmdb dataset with rotten tomatoes using the title column
tmdb_rotten_df =  pd.merge(
    tmdb_df,
    rotten_df,
    on="title",
    how="inner"
)

print("TMDB:", len(tmdb_df))
print("Rotten:", len(rotten_df))
print("Merged:", len(tmdb_rotten_df))

display(tmdb_rotten_df.head())

In [ ]:
# testing merging rotten tomatoes dataset with numbers using the title column
rotten_numbers_df = pd.merge(
    rotten_df,
    numbers_df,
    on="title",
    how="inner"
)

print("The Numbers:", len(numbers_df))
print("Rotten:", len(rotten_df))
print("Merged:", len(tmdb_rotten_df))

display(rotten_numbers_df.head())

In [ ]:
# merging all
final_df = pd.merge(
    rotten_numbers_df,
    tmdb_df,
    on="title",
    how="inner"
)
print("Merged:", len(final_df))

display(final_df.head())

In [ ]:
print('length of tmdb',len(tmdb_df))
print('length of rotten',len(rotten_df))
print('length of numbers',len(numbers_df))

print("length of tmdb_numbers_df:", len(tmdb_numbers_df))
print('length of tmdb_rotten_df:', len(tmdb_rotten_df))
print('length of rotten_numbers_df:', len(rotten_numbers_df))
print('length of all merged datasets:', len(final_df))

In [ ]:
# check in tmdb lines where title and original title are different
diff_tmdb_df = tmdb_df[tmdb_df['title'] != tmdb_df['original_title']]

display(diff_tmdb_df)

the merge is correctly done on the title only

In [ ]:
import re
import unicodedata

def normalize_title(title):
    title = str(title).lower()

    # Remove accents
    title = unicodedata.normalize('NFKD', title)
    title = ''.join(c for c in title if not unicodedata.combining(c))

    # Remove punctuation
    title = re.sub(r'[^a-z0-9\s]', '', title)

    # Remove extra spaces
    title = re.sub(r'\s+', ' ', title).strip()

    return title

In [ ]:
tmdb_df['title_normalized'] = tmdb_df['title'].apply(normalize_title)
rotten_df['title_normalized'] = rotten_df['title'].apply(normalize_title)
numbers_df['title_normalized'] = numbers_df['title'].apply(normalize_title)

In [ ]:
numbers_df[['title','title_normalized']]


In [ ]:
tmdb_df["title_normalized"]

In [ ]:
# pegasus 3 3 --> 3 was inside the chinese translation
# should we automatically eleminate whats after the parantheses (translation) or treat it manually ?
display(
    tmdb_df[tmdb_df['title_normalized'].str.startswith('pegasus')][
        ['title', 'title_normalized']
    ]
)
#  the movie actually exists in tmdb_df !

In [ ]:
tmdb_rotten_df_test = pd.merge(
    tmdb_df,
    rotten_df,
    on='title_normalized',
    how='inner',
    suffixes=('_tmdb', '_rotten')
)
final_df_normalized= pd.merge(
  tmdb_rotten_df_test,
  numbers_df,
  on='title_normalized',
  how='inner'
)
print("length of final df:", len(final_df_normalized))

In [ ]:
# lets see if pegasus 3 was preserved
print(final_df_normalized[final_df_normalized['title'] == 'pegasus 3'])

it was removed !!  we need to apply a specific type of matching

In [ ]:
#  why is final_df and final_df_normalized different in 2 titles
# displaying titles that exist in final_df_normalized and dont exist in final_df
diff = final_df_normalized[
    ~final_df_normalized['title'].isin(final_df['title'])
]

display(diff[['title']])

In [ ]:
# displaying the rows of those two differing titles
result = final_df_normalized[
    final_df_normalized['title'].isin(diff['title'])
]

display(result[['title', 'title_tmdb', 'title_rotten','title_normalized']])

In [ ]:
# display in each df the rows where title is in diff
tmdb_result= tmdb_df[
  tmdb_df['title'].isin(diff['title'])
]
display(tmdb_result['title'])

In [ ]:
# display in each df the rows where title is in diff
numbers_result= numbers_df[
  numbers_df['title'].isin(diff['title'])
]
display(numbers_result['title'])

In [ ]:
# display in each df the rows where title is in diff
rotten_result= rotten_df[
  rotten_df['title'].isin(diff['title'])
]
display(rotten_result['title'])

In [ ]:
unmatched = tmdb_df[
    ~tmdb_df['title_normalized'].isin(numbers_df['title_normalized'])
]

print(len(unmatched))
display(unmatched[['title', 'title_normalized']])

In [ ]:
exact_matches = pd.merge(
    tmdb_df,
    numbers_df,
    on='title_normalized',
    how='inner',
    suffixes=('_tmdb', '_numbers')
)

print(len(exact_matches))

In [ ]:
from rapidfuzz import process, fuzz

unmatched_tmdb = tmdb_df[
    ~tmdb_df['title_normalized'].isin(numbers_df['title_normalized'])
].copy()

In [ ]:
numbers_titles = numbers_df['title_normalized'].tolist()

matches = []

for title in unmatched_tmdb['title_normalized']:
    match = process.extractOne(
        title,
        numbers_titles,
        scorer=fuzz.ratio
    )

    matches.append({
        'tmdb_title': title, # here its the normalized title of tmdb_df
        'numbers_title': match[0], # same
        'similarity': match[1]
    })

fuzzy_matches = pd.DataFrame(matches)

display(
    fuzzy_matches.sort_values(
        'similarity',
        ascending=False
    )
)

as we can see only the 

In [ ]:
numbers_titles = numbers_df['title_normalized'].tolist()

matches = []

for _, tmdb_row in unmatched_tmdb.iterrows():

    tmdb_normalized = tmdb_row['title_normalized']

    match = process.extractOne(
        tmdb_normalized,
        numbers_titles,
        scorer=fuzz.ratio
    )

    # Find the corresponding Numbers row
    numbers_row = numbers_df[
        numbers_df['title_normalized'] == match[0]
    ].iloc[0]

    matches.append({
        'tmdb_original': tmdb_row['title'],
        'tmdb_normalized': tmdb_normalized,
        'numbers_original': numbers_row['title'],
        'numbers_normalized': numbers_row['title_normalized'],
        'similarity': match[1]
    })

fuzzy_matches = pd.DataFrame(matches)

display(
    fuzzy_matches.sort_values(
        'similarity',
        ascending=False
    )
)

lets check if other columns could help us see if the 215th movie is actually the same or not even if the similarity is high

In [ ]:
tmdb_df.columns

In [ ]:
numbers_df.columns

In [ ]:
# retrieve the url link for the movie: Torrente, Presidente
# to see if its the same movie than Torrente for Presidente
display(
    numbers_df[
        numbers_df['title'] == 'Torrente, Presidente'
    ]
)

after checking in the browner its actually the same movie!! 

but the tmdb database had an error in its title and added the "for" which doesnt exist originnaly

but the original title in tmdb can sometimes be better because it doesnt contain the "for"

the problem is that the original title isnt always in english!!

In [ ]:
display(
    numbers_df[
        numbers_df['title'] == 'Passenger'
    ]
)

In [ ]:
# Star Wars: The Mandalorian and Grogu
display(
    numbers_df[
        numbers_df['title'] == 'Star Wars: The Mandalorian and Grogu'
    ]
)

79 à 17 are the same movies